## Handling the Extract and Transform part

In [31]:
import os
import numpy as np
import pandas as pd
from datetime import date

import chardet
import seaborn as sns

In [2]:
def detect_encoding(file):
    detector = chardet.universaldetector.UniversalDetector()
    with open(file, "rb") as f:
        for line in f:
            detector.feed(line)
            if detector.done:
                break
    detector.close()
    return detector.result

In [3]:
# loc = "C:\Offline_Docs\Personal\DT_CS_2023\DE_CS_202309\Project\Case_Study_202309_Data"
loc = r"F:\github\DE_CS_202309\Project\Case_Study_202309_Data"

enc_list = {}

In [4]:
# Detect Character Encoding

file = loc+'\\'+ os.listdir(loc)[0]
encoding = detect_encoding(file)['encoding']
# print("Encoding:", encoding['encoding'])
encoding

'ascii'

In [5]:
count = 0
filenames = []

for file in os.listdir(loc):
  if file.endswith(".csv"):
    filenames.append(file)
    count+=1
    encoding = detect_encoding(loc+'\\'+file)['encoding']
    enc_list[file]=encoding

print(f'Total {count} CSV files present')

Total 130 CSV files present


In [6]:
df_enc_lst= pd.DataFrame.from_dict(enc_list,orient='Index')
df_enc_lst.reset_index(inplace=True)
df_enc_lst.columns=['File_Name','Encoding']
df_enc_lst.head()

,File_Name,Encoding
0,201901_Orders_2019_02_01_03_10_55.csv,ascii
1,201901_Orders_2019_02_04_15_41_32.csv,ascii
2,201902_Orders_2019_03_04_02_26_31.csv,Windows-1252
3,201903_Orders_2019_04_01_20_39_17.csv,Windows-1252
4,201903_Orders_2019_04_04_08_51_54.csv,ISO-8859-1


## Analyze the File names

In [7]:
df_enc_lst["Orders_For"] = df_enc_lst["File_Name"].str.split("_",n=1, expand=True)[0]
df_enc_lst["Date_File_Load"] = df_enc_lst["File_Name"].str.split("_",expand=True,n=2)[2].str.split(".",expand=True)[0]
df_enc_lst['Date_File_Load']= pd.to_datetime(df_enc_lst["Date_File_Load"],format="%Y_%m_%d_%H_%M_%S")
df_enc_lst["Load_Day"]= df_enc_lst["Date_File_Load"].dt.day_name()

In [50]:
df_enc_lst.head()

,File_Name,Encoding,Orders_For,Date_File_Load,Day_of_Drop
0,201901_Orders_2019_02_01_03_10_55.csv,ascii,201901,2019-02-01 03:10:55,Friday
1,201901_Orders_2019_02_04_15_41_32.csv,ascii,201901,2019-02-04 15:41:32,Monday
2,201902_Orders_2019_03_04_02_26_31.csv,Windows-1252,201902,2019-03-04 02:26:31,Monday
3,201903_Orders_2019_04_01_20_39_17.csv,Windows-1252,201903,2019-04-01 20:39:17,Monday
4,201903_Orders_2019_04_04_08_51_54.csv,ISO-8859-1,201903,2019-04-04 08:51:54,Thursday


In [55]:
df_enc_lst.groupby(by='Day_of_Drop').count()

,File_Name,Encoding,Orders_For,Date_File_Load
Day_of_Drop,,,,
Friday,19,19,19,19
Monday,20,20,20,20
Saturday,14,14,14,14
Sunday,21,21,21,21
Thursday,20,20,20,20
Tuesday,18,18,18,18
Wednesday,18,18,18,18


## File Encoding List

In [9]:
df_enc_lst['Encoding'].value_counts()

Encoding
Windows-1252    68
ISO-8859-1      53
ascii            9
Name: count, dtype: int64

## Create consolidated File

In [10]:
df1 = pd.read_csv(loc+'\\'+ os.listdir(loc)[0], delimiter='|', encoding='utf8')
df1.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,7981,CA-2019-103800,04-01-2019,08-01-2019,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,...,77095,Central,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",0,0,0.2,5.5512
1,740,CA-2019-112326,05-01-2019,09-01-2019,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,0,0,0.2,4.2717
2,741,CA-2019-112326,05-01-2019,09-01-2019,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,0,0,0.2,-64.7748
3,742,CA-2019-112326,05-01-2019,09-01-2019,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,...,60540,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,0,0,0.8,-5.4870
4,1760,CA-2019-141817,06-01-2019,13-01-2019,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,...,19143,East,OFF-AR-10003478,Office Supplies,Art,Avery Hi-Liter EverBold Pen Style Fluorescent ...,0,0,0.2,4.8840


In [11]:
mega_df = pd.DataFrame(columns=df1.columns)
mega_df

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit


In [12]:
type(np.count_nonzero(filenames))

int

In [21]:
col_cnt = {}

for row, file in enumerate(filenames):
  tmp_df = pd.read_csv(loc+'\\'+ file, delimiter='|',encoding=enc_list[file])
  # pd.concat([mega_df, tmp_df])
  cnt_cols = np.count_nonzero(tmp_df.columns)
  col_cnt[file] = np.count_nonzero(tmp_df.columns)
  if cnt_cols != 21:
    print(f'Total Columns, {np.count_nonzero(tmp_df.columns)}, Filename - {file}')
  else:
    if row != np.count_nonzero(filenames)-1:
      mega_df = pd.concat([mega_df,tmp_df])


Total Columns, 20, Filename - 202206_Orders_2022_07_05_18_29_34.csv


In [15]:
mega_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21446 entries, 0 to 446
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         21446 non-null  object 
 1   Order ID       21446 non-null  object 
 2   Order Date     21446 non-null  object 
 3   Ship Date      21446 non-null  object 
 4   Ship Mode      21446 non-null  object 
 5   Customer ID    21446 non-null  object 
 6   Customer Name  21446 non-null  object 
 7   Segment        21446 non-null  object 
 8   Country        21446 non-null  object 
 9   City           21446 non-null  object 
 10  State          21446 non-null  object 
 11  Postal Code    21446 non-null  object 
 12  Region         21446 non-null  object 
 13  Product ID     21446 non-null  object 
 14  Category       21446 non-null  object 
 15  Sub-Category   21446 non-null  object 
 16  Product Name   21446 non-null  object 
 17  Sales          21446 non-null  object 
 18  Quantity     

In [16]:
mega_df.drop(columns=['Row ID'], inplace=True)

In [17]:
mega_df[['Sales','Quantity']] = mega_df[['Sales','Quantity']].astype(float)

In [18]:
mega_df.head()

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,CA-2019-103800,04-01-2019,08-01-2019,Standard Class,DP-13000,Darren Powers,Consumer,United States,Houston,Texas,77095,Central,OFF-PA-10000174,Office Supplies,Paper,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",0.0,0.0,0.2,5.5512
1,CA-2019-112326,05-01-2019,09-01-2019,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540,Central,OFF-LA-10003223,Office Supplies,Labels,Avery 508,0.0,0.0,0.2,4.2717
2,CA-2019-112326,05-01-2019,09-01-2019,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540,Central,OFF-ST-10002743,Office Supplies,Storage,SAFCO Boltless Steel Shelving,0.0,0.0,0.2,-64.7748
3,CA-2019-112326,05-01-2019,09-01-2019,Standard Class,PO-19195,Phillina Ober,Home Office,United States,Naperville,Illinois,60540,Central,OFF-BI-10004094,Office Supplies,Binders,GBC Standard Plastic Binding Systems Combs,0.0,0.0,0.8,-5.4870
4,CA-2019-141817,06-01-2019,13-01-2019,Standard Class,MB-18085,Mick Brown,Consumer,United States,Philadelphia,Pennsylvania,19143,East,OFF-AR-10003478,Office Supplies,Art,Avery Hi-Liter EverBold Pen Style Fluorescent ...,0.0,0.0,0.2,4.8840


In [19]:
# mega_df.to_csv(loc+"\\"+'file.csv')

## Analyse Data from File Names

In [27]:
for x in col_cnt:
  if df_enc_lst["File_Name"] == x:
    df_enc_lst["Col_Count"] = col_cnt[x]
  # print(f'{x} -- {col_cnt[x]}')

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().